In [3]:
# Importation des modules
import socket       # Pour la communication réseau (TCP)
import secrets      # Pour la génération sécurisée de nombres aléatoires
import hashlib      # Pour le hachage SHA-256

# Configuration de l'adresse et du port du serveur
hote = '127.0.0.1'  # Adresse locale (localhost)
port = 65432        # Port d'écoute du serveur

# ------------------------
# --- Question 1 : Établissement du secret partagé via Diffie-Hellman ---
# Bob génère une clé privée, calcule sa clé publique, puis reçoit celle d'Alice.
# Il utilise sa clé privée et la clé publique d'Alice pour calculer un secret partagé.

# 1. Génération de la paire de clés Diffie-Hellman côté Bob
premier = 0xFFFFFFFB
generateur = 2 #Vu avec le prof
cle_privee_bob = secrets.randbelow(premier)
cle_publique_bob = pow(generateur, cle_privee_bob, premier)

# Création du socket serveur TCP
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as serveur:
    serveur.bind((hote, port))
    serveur.listen()
    print("Serveur Bob en attente de connexion...")

    connexion, adresse = serveur.accept()
    with connexion:
        print(f"Connecté à {adresse}")

        # Réception de la clé publique d'Alice
        cle_publique_alice = int(connexion.recv(4096).decode())

        # Envoi de la clé publique de Bob à Alice
        connexion.sendall(str(cle_publique_bob).encode())

        # Calcul du secret partagé (même clé de session des deux côtés)
        secret_partage = pow(cle_publique_alice, cle_privee_bob, premier)
        #secret_partage = (pow(cle_publique_alice, cle_privee_bob, premier) + 1) % premier  # Cas d'erreur
        secret_partage_octets = secret_partage.to_bytes(16, byteorder='big')


        # ------------------------
        # --- Question 2 : Déchiffrement de la clé AES ---
        # Bob reçoit la clé AES chiffrée par Alice via XOR avec le secret partagé.

        donnees = connexion.recv(4096)
        cle_aes_chiffree = donnees[:16]           # Clé AES chiffrée (16 octets)
        hash_recu = donnees[16:48]                # SHA-256 de la clé AES (32 octets)

        # Déchiffrement de la clé AES avec le secret partagé
        cle_aes = bytes([c ^ s for c, s in zip(cle_aes_chiffree, secret_partage_octets)])

        # ------------------------
        # --- Question 3 : Vérification d'intégrité avec SHA-256 ---
        # Bob vérifie si la clé AES reçue n’a pas été altérée.

        if hashlib.sha256(cle_aes).digest() == hash_recu:
            print("Intégrité confirmée : clé AES correcte.")
        else:
            print("Intégrité NON confirmée : clé AES corrompue !")

        # ------------------------
        # --- Question 4 : Confirmation par retour de message ---
        # Bob envoie un message chiffré avec la clé AES pour confirmer qu’il l’a bien reçue.

        message = b"CBON".ljust(16, b'\x00')
        message_chiffre = bytes([m ^ k for m, k in zip(message, cle_aes)])

        connexion.sendall(message_chiffre)
        print("Message de confirmation envoyé à Alice.")


Serveur Bob en attente de connexion...
Connecté à ('127.0.0.1', 47526)
Intégrité confirmée : clé AES correcte.
Message de confirmation envoyé à Alice.
